# BERT Raw Benchmark

Notebook này đánh giá `google-bert/bert-base-uncased` **không distill / không fine-tune encoder** trên toàn bộ 9 benchmark của paper.

- Classification: frozen BERT embeddings + Logistic Regression, metric chính là macro-F1.
- Pair Classification: cosine similarity giữa hai embedding, metric chính là Average Precision.
- STS: cosine similarity so với human score, metric chính là Spearman correlation.

Kết quả được lưu vào `bert_raw.csv`.

In [1]:
# Kiểm tra GPU trên Colab. Cell này vẫn chạy được nếu không có nvidia-smi.
import os
os.system('nvidia-smi || true')

0

In [2]:
# Mount Google Drive nếu đang chạy trên Colab.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Not running inside Google Colab; skip Drive mount.')

Mounted at /content/drive


In [3]:
# Tìm project root. Nếu bạn chạy notebook ngay trong repo thì cell này sẽ giữ nguyên cwd.
# Nếu chạy theo setup cũ trên Drive, cell này cũng thử unzip các file quen thuộc.
from pathlib import Path
import os
import zipfile

def is_project_root(path: Path) -> bool:
    return (path / 'requirements.txt').exists() and (path / 'data' / 'multi-data').exists()

candidate_roots = [
    Path.cwd(),
    Path('/content/AAAI-TALAS'),
    Path('/content/AAAI-TALAS-new_idea_2'),
    Path('/content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS'),
    Path('/content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS-new_idea_2'),
]

project_root = next((p for p in candidate_roots if is_project_root(p)), None)

if project_root is None and IN_COLAB:
    zip_candidates = [
        Path('/content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS.zip'),
        Path('/content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS-new_idea_2.zip'),
    ]
    for zip_path in zip_candidates:
        if zip_path.exists():
            print(f'Unzipping {zip_path} to /content ...')
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall('/content')
            break
    project_root = next((p for p in candidate_roots if is_project_root(p)), None)

if project_root is None:
    raise FileNotFoundError(
        'Could not find project root. Please open this notebook inside AAAI-TALAS, '
        'or place the repo/zip under /content/drive/MyDrive/AAAI_TALAS/.'
    )

os.chdir(project_root)
print('Project root:', Path.cwd())

Unzipping /content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS-new_idea_2.zip to /content ...
Project root: /content/AAAI-TALAS-new_idea_2


In [4]:
# Không cài package vào global environment.
# Nếu Colab thiếu package nào, notebook tạo venv cục bộ .venv_bert_raw rồi add site-packages vào sys.path.
import importlib.util
import subprocess
import sys
import os
import json
from pathlib import Path

required = {
    'torch': 'torch',
    'transformers': 'transformers',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'sklearn': 'scikit-learn',
    'scipy': 'scipy',
    'tqdm': 'tqdm',
}

missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
venv_dir = Path.cwd() / '.venv_bert_raw'

if missing:
    print('Missing packages:', missing)
    if not venv_dir.exists():
        subprocess.check_call([sys.executable, '-m', 'venv', str(venv_dir)])
    pip_path = venv_dir / ('Scripts/pip.exe' if os.name == 'nt' else 'bin/pip')
    python_path = venv_dir / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    subprocess.check_call([str(pip_path), 'install', '--upgrade', 'pip'])
    subprocess.check_call([str(pip_path), 'install', *missing])
    site_packages = subprocess.check_output([
        str(python_path), '-c',
        'import site, json; print(json.dumps(site.getsitepackages()))'
    ], text=True)
    for path in json.loads(site_packages):
        if path not in sys.path:
            sys.path.insert(0, path)
else:
    print('All required packages are already available.')

All required packages are already available.


In [5]:
# Imports và cấu hình chung.
from pathlib import Path
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

SEED = 42
MODEL_NAME = 'google-bert/bert-base-uncased'
BATCH_SIZE = 64
MAX_LENGTH = 256
OUTPUT_CSV = Path('bert_raw.csv')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [6]:
# Load BERT raw. Encoder được giữ nguyên, không fine-tune.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print('Loaded:', MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: google-bert/bert-base-uncased


In [7]:
# Danh sách benchmark giống protocol của paper/TALAS.
CLASSIFICATION_TASKS = [
    {'dataset': 'BANKING77', 'train': 'data/multi-data/banking_train.csv', 'validation': 'data/multi-data/banking77_validation.csv', 'test': 'data/multi-data/banking77_test.csv', 'domain': 'out'},
    {'dataset': 'Emotion', 'train': 'data/multi-data/emotion_train.csv', 'validation': 'data/multi-data/emotion_validation.csv', 'test': 'data/multi-data/emotion_test.csv', 'domain': 'in'},
    {'dataset': 'TweetEval-Sentiment', 'train': 'data/multi-data/tweet_train.csv', 'validation': 'data/multi-data/tweet_validation.csv', 'test': 'data/multi-data/tweet_test.csv', 'domain': 'out'},
]

PAIR_CLASSIFICATION_TASKS = [
    {'dataset': 'MRPC', 'validation': 'data/multi-data/mrpc_validation.csv', 'test': 'data/multi-data/mrpc_test.csv', 'domain': 'out'},
    {'dataset': 'SciTail', 'validation': 'data/multi-data/scitail_validation.csv', 'test': 'data/multi-data/scitail_test.csv', 'domain': 'out'},
    {'dataset': 'WiC', 'validation': 'data/multi-data/wic_validation.csv', 'test': 'data/multi-data/wic_test.csv', 'domain': 'in'},
]

STS_TASKS = [
    {'dataset': 'SICK', 'validation': 'data/multi-data/sick_validation.csv', 'test': 'data/multi-data/sick_test.csv', 'domain': 'out'},
    {'dataset': 'STS12', 'validation': 'data/multi-data/sts12_validation.csv', 'test': 'data/multi-data/sts12_test.csv', 'domain': 'out'},
    {'dataset': 'STS-B', 'validation': 'data/multi-data/stsb_validation.csv', 'test': 'data/multi-data/stsb_test.csv', 'domain': 'in'},
]

all_paths = []
for task in CLASSIFICATION_TASKS:
    all_paths.extend([task['train'], task['validation'], task['test']])
for task in PAIR_CLASSIFICATION_TASKS + STS_TASKS:
    all_paths.extend([task['validation'], task['test']])

missing_files = [p for p in all_paths if not Path(p).exists()]
if missing_files:
    raise FileNotFoundError('Missing benchmark files:\n' + '\n'.join(missing_files))

print('Found all benchmark files.')

Found all benchmark files.


In [8]:
# Helper: encode text thành CLS embedding giống evaluator hiện có trong src/evaluation/evaluation_automodel.py.
@torch.no_grad()
def encode_texts(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    embeddings = []
    texts = [str(t) for t in texts]
    for start in tqdm(range(0, len(texts), batch_size), leave=False):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            max_length=max_length,
            truncation=True,
            padding=True,
            return_tensors='pt',
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=(device.type == 'cuda')):
            output = model(**encoded)
            cls_embedding = output.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.detach().float().cpu())
    return torch.cat(embeddings, dim=0).numpy()

EMBEDDING_CACHE = {}

def encode_column(cache_key, texts):
    if cache_key not in EMBEDDING_CACHE:
        EMBEDDING_CACHE[cache_key] = encode_texts(texts)
    return EMBEDDING_CACHE[cache_key]

def pct(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return np.nan
    return float(value) * 100.0

def safe_corr(fn, x, y):
    result = fn(x, y)
    return float(result.statistic if hasattr(result, 'statistic') else result[0])

In [9]:
# Evaluation functions.
def evaluate_classification(task, split):
    train_df = pd.read_csv(task['train'])
    eval_df = pd.read_csv(task[split])

    print(f"\n[Classification] {task['dataset']} / {split}")
    x_train = encode_column((task['train'], 'text'), train_df['text'].tolist())
    x_eval = encode_column((task[split], 'text'), eval_df['text'].tolist())
    y_train = train_df['label'].astype(int).to_numpy()
    y_eval = eval_df['label'].astype(int).to_numpy()

    clf = LogisticRegression(random_state=SEED, n_jobs=1, max_iter=200, verbose=0)
    clf.fit(x_train, y_train)
    pred = clf.predict(x_eval)

    accuracy = accuracy_score(y_eval, pred)
    macro_f1 = f1_score(y_eval, pred, average='macro')
    return {
        'split': split,
        'task_group': 'Classification',
        'dataset': task['dataset'],
        'domain': task['domain'],
        'primary_metric': 'macro_f1',
        'primary_score': pct(macro_f1),
        'accuracy': pct(accuracy),
        'macro_f1': pct(macro_f1),
        'average_precision': np.nan,
        'precision': np.nan,
        'recall': np.nan,
        'best_threshold': np.nan,
        'spearman': np.nan,
        'pearson': np.nan,
        'n_train': len(train_df),
        'n_eval': len(eval_df),
        'model': MODEL_NAME,
    }

def evaluate_pair_classification(task, split):
    df = pd.read_csv(task[split])
    print(f"\n[Pair Classification] {task['dataset']} / {split}")
    emb1 = encode_column((task[split], 'sentence1'), df['sentence1'].tolist())
    emb2 = encode_column((task[split], 'sentence2'), df['sentence2'].tolist())
    labels = df['label'].astype(int).to_numpy()

    scores = (F.cosine_similarity(torch.tensor(emb1), torch.tensor(emb2)).numpy() + 1.0) / 2.0
    average_precision = average_precision_score(labels, scores)

    best_acc, best_thr = 0.0, 0.0
    for thr in np.linspace(0, 1, 200):
        pred = (scores >= thr).astype(int)
        acc = accuracy_score(labels, pred)
        if acc > best_acc:
            best_acc, best_thr = acc, float(thr)
    pred = (scores >= best_thr).astype(int)

    return {
        'split': split,
        'task_group': 'Pair Classification',
        'dataset': task['dataset'],
        'domain': task['domain'],
        'primary_metric': 'average_precision',
        'primary_score': pct(average_precision),
        'accuracy': pct(best_acc),
        'macro_f1': pct(f1_score(labels, pred, average='macro')),
        'average_precision': pct(average_precision),
        'precision': pct(precision_score(labels, pred, average='macro', zero_division=0)),
        'recall': pct(recall_score(labels, pred, average='macro', zero_division=0)),
        'best_threshold': best_thr,
        'spearman': np.nan,
        'pearson': np.nan,
        'n_train': np.nan,
        'n_eval': len(df),
        'model': MODEL_NAME,
    }

def evaluate_sts(task, split):
    df = pd.read_csv(task[split])
    print(f"\n[STS] {task['dataset']} / {split}")
    emb1 = encode_column((task[split], 'sentence1'), df['sentence1'].tolist())
    emb2 = encode_column((task[split], 'sentence2'), df['sentence2'].tolist())
    labels = df['score'].astype(float).to_numpy()

    cosine = F.cosine_similarity(torch.tensor(emb1), torch.tensor(emb2)).numpy()
    pred_scores = (cosine + 1.0) * 2.5
    spearman = safe_corr(spearmanr, pred_scores, labels)
    pearson = safe_corr(pearsonr, pred_scores, labels)

    return {
        'split': split,
        'task_group': 'STS',
        'dataset': task['dataset'],
        'domain': task['domain'],
        'primary_metric': 'spearman',
        'primary_score': pct(spearman),
        'accuracy': np.nan,
        'macro_f1': np.nan,
        'average_precision': np.nan,
        'precision': np.nan,
        'recall': np.nan,
        'best_threshold': np.nan,
        'spearman': pct(spearman),
        'pearson': pct(pearson),
        'n_train': np.nan,
        'n_eval': len(df),
        'model': MODEL_NAME,
    }

In [10]:
# Chạy toàn bộ 9 benchmark trên validation và test.
rows = []

for split in ['validation', 'test']:
    for task in CLASSIFICATION_TASKS:
        rows.append(evaluate_classification(task, split))
    for task in PAIR_CLASSIFICATION_TASKS:
        rows.append(evaluate_pair_classification(task, split))
    for task in STS_TASKS:
        rows.append(evaluate_sts(task, split))

results = pd.DataFrame(rows)
results


[Classification] BANKING77 / validation


  0%|          | 0/157 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]


[Classification] Emotion / validation


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


...

  0%|          | 0/49 [00:00<?, ?it/s]


[STS] STS-B / test


  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

,split,task_group,dataset,domain,primary_metric,primary_score,accuracy,macro_f1,average_precision,precision,recall,best_threshold,spearman,pearson,n_train,n_eval,model
0,validation,Classification,BANKING77,out,macro_f1,99.096947,99.000000,99.096947,NaN,NaN,NaN,NaN,NaN,NaN,10003.0,1000,google-bert/bert-base-uncased
1,validation,Classification,Emotion,in,macro_f1,47.980774,58.802817,47.980774,NaN,NaN,NaN,NaN,NaN,NaN,15956.0,1988,google-bert/bert-base-uncased
2,validation,Classification,TweetEval-Sentiment,out,macro_f1,70.274427,70.044665,70.274427,NaN,NaN,NaN,NaN,NaN,NaN,26732.0,23732,google-bert/bert-base-uncased
3,validation,Pair Classification,MRPC,out,average_precision,83.987827,69.607843,59.209211,83.987827,63.279493,59.023089,0.934673,NaN,NaN,NaN,408,google-bert/bert-base-uncased
4,validation,Pair Classification,SciTail,out,average_precision,81.963907,72.085890,72.073013,81.963907,72.162070,72.106597,0.924623,NaN,NaN,NaN,1304,google-bert/bert-base-uncased
5,validation,Pair Classification,WiC,in,average_precision,61.584559,58.620690,58.251455,61.584559,58.936849,58.620690,0.929648,NaN,NaN,NaN,638,google-bert/bert-base-uncased
6,validation,STS,SICK,out,spearman,44.570635,NaN,NaN,NaN,NaN,NaN,NaN,44.570635,39.417145,NaN,500,google-bert/bert-base-uncased
7,validation,STS,STS12,out,spearman,21.760066,NaN,NaN,NaN,NaN,NaN,NaN,21.760066,14.214238,NaN,734,google-bert/bert-base-uncased
8,validation,STS,STS-B,in,spearman,31.738028,NaN,NaN,NaN,NaN,NaN,NaN,31.738028,29.156470,NaN,1500,google-bert/bert-base-uncased
9,test,Classification,BANKING77,out,macro_f1,85.262217,85.305592,85.262217,NaN,NaN,NaN,NaN,NaN,NaN,10003.0,3076,google-bert/bert-base-uncased


In [11]:
# Thêm các dòng summary giống paper: Avg-In, Avg-Out, Avg-All.
summary_rows = []
for split, split_df in results.groupby('split', sort=False):
    for label, domain_filter in [
        ('Avg-In', split_df['domain'].eq('in')),
        ('Avg-Out', split_df['domain'].eq('out')),
        ('Avg-All', split_df['domain'].isin(['in', 'out'])),
    ]:
        subset = split_df[domain_filter]
        summary_rows.append({
            'split': split,
            'task_group': 'Summary',
            'dataset': label,
            'domain': label.replace('Avg-', '').lower(),
            'primary_metric': 'mean_primary_score',
            'primary_score': subset['primary_score'].mean(),
            'accuracy': np.nan,
            'macro_f1': np.nan,
            'average_precision': np.nan,
            'precision': np.nan,
            'recall': np.nan,
            'best_threshold': np.nan,
            'spearman': np.nan,
            'pearson': np.nan,
            'n_train': np.nan,
            'n_eval': subset['n_eval'].sum(),
            'model': MODEL_NAME,
        })

results_with_summary = pd.concat([results, pd.DataFrame(summary_rows)], ignore_index=True)

display_cols = ['split', 'task_group', 'dataset', 'domain', 'primary_metric', 'primary_score', 'accuracy', 'macro_f1', 'average_precision', 'spearman', 'pearson', 'n_eval']
display(results_with_summary[display_cols].round(4))

,split,task_group,dataset,domain,primary_metric,primary_score,accuracy,macro_f1,average_precision,spearman,pearson,n_eval
0,validation,Classification,BANKING77,out,macro_f1,99.0969,99.0000,99.0969,NaN,NaN,NaN,1000
1,validation,Classification,Emotion,in,macro_f1,47.9808,58.8028,47.9808,NaN,NaN,NaN,1988
2,validation,Classification,TweetEval-Sentiment,out,macro_f1,70.2744,70.0447,70.2744,NaN,NaN,NaN,23732
3,validation,Pair Classification,MRPC,out,average_precision,83.9878,69.6078,59.2092,83.9878,NaN,NaN,408
4,validation,Pair Classification,SciTail,out,average_precision,81.9639,72.0859,72.0730,81.9639,NaN,NaN,1304
5,validation,Pair Classification,WiC,in,average_precision,61.5846,58.6207,58.2515,61.5846,NaN,NaN,638
6,validation,STS,SICK,out,spearman,44.5706,NaN,NaN,NaN,44.5706,39.4171,500
7,validation,STS,STS12,out,spearman,21.7601,NaN,NaN,NaN,21.7601,14.2142,734
8,validation,STS,STS-B,in,spearman,31.7380,NaN,NaN,NaN,31.7380,29.1565,1500
9,test,Classification,BANKING77,out,macro_f1,85.2622,85.3056,85.2622,NaN,NaN,NaN,3076


In [12]:
# Bảng nhanh: primary score theo dataset/split để dễ nhìn performance của BERT raw.
quick_view = results.pivot(index='dataset', columns='split', values='primary_score').loc[
    ['BANKING77', 'TweetEval-Sentiment', 'Emotion', 'MRPC', 'SciTail', 'WiC', 'SICK', 'STS12', 'STS-B']
]
quick_view.loc['Avg-In'] = results[results['domain'].eq('in')].groupby('split')['primary_score'].mean()
quick_view.loc['Avg-Out'] = results[results['domain'].eq('out')].groupby('split')['primary_score'].mean()
quick_view.loc['Avg-All'] = results.groupby('split')['primary_score'].mean()

display(quick_view.round(2))

split,test,validation
dataset,,
BANKING77,85.26,99.10
TweetEval-Sentiment,67.61,70.27
Emotion,47.63,47.98
MRPC,77.36,83.99
SciTail,67.75,81.96
WiC,59.23,61.58
SICK,42.42,44.57
STS12,21.54,21.76
STS-B,20.30,31.74


In [13]:
# Save CSV cuối cùng.
results_with_summary.to_csv(OUTPUT_CSV, index=False)
print(f'Saved results to: {OUTPUT_CSV.resolve()}')

# In vài dòng đầu để kiểm tra nhanh.
display(pd.read_csv(OUTPUT_CSV).head(12).round(4))

Saved results to: /content/AAAI-TALAS-new_idea_2/bert_raw.csv


,split,task_group,dataset,domain,primary_metric,primary_score,accuracy,macro_f1,average_precision,precision,recall,best_threshold,spearman,pearson,n_train,n_eval,model
0,validation,Classification,BANKING77,out,macro_f1,99.0969,99.0000,99.0969,NaN,NaN,NaN,NaN,NaN,NaN,10003.0,1000,google-bert/bert-base-uncased
1,validation,Classification,Emotion,in,macro_f1,47.9808,58.8028,47.9808,NaN,NaN,NaN,NaN,NaN,NaN,15956.0,1988,google-bert/bert-base-uncased
2,validation,Classification,TweetEval-Sentiment,out,macro_f1,70.2744,70.0447,70.2744,NaN,NaN,NaN,NaN,NaN,NaN,26732.0,23732,google-bert/bert-base-uncased
3,validation,Pair Classification,MRPC,out,average_precision,83.9878,69.6078,59.2092,83.9878,63.2795,59.0231,0.9347,NaN,NaN,NaN,408,google-bert/bert-base-uncased
4,validation,Pair Classification,SciTail,out,average_precision,81.9639,72.0859,72.0730,81.9639,72.1621,72.1066,0.9246,NaN,NaN,NaN,1304,google-bert/bert-base-uncased
5,validation,Pair Classification,WiC,in,average_precision,61.5846,58.6207,58.2515,61.5846,58.9368,58.6207,0.9296,NaN,NaN,NaN,638,google-bert/bert-base-uncased
6,validation,STS,SICK,out,spearman,44.5706,NaN,NaN,NaN,NaN,NaN,NaN,44.5706,39.4171,NaN,500,google-bert/bert-base-uncased
7,validation,STS,STS12,out,spearman,21.7601,NaN,NaN,NaN,NaN,NaN,NaN,21.7601,14.2142,NaN,734,google-bert/bert-base-uncased
8,validation,STS,STS-B,in,spearman,31.7380,NaN,NaN,NaN,NaN,NaN,NaN,31.7380,29.1565,NaN,1500,google-bert/bert-base-uncased
9,test,Classification,BANKING77,out,macro_f1,85.2622,85.3056,85.2622,NaN,NaN,NaN,NaN,NaN,NaN,10003.0,3076,google-bert/bert-base-uncased
